# Phase 3 — Analyze
## 04 — Cross Functional Analysis

## Objective

Integrate sales performance, inventory position, and Sell-Out Allowance (SOA) information at product level to identify commercially actionable relationships across:

- Product demand
- Inventory exposure
- Profitability
- Promotional / SOA support

The analysis focuses on identifying products requiring commercial action, such as excess stock, weak demand, margin pressure, or available SOA support.

This notebook uses governed processed datasets created in earlier phases and does not repeat standalone Sales, Inventory, or SOA analysis.

In [1]:
# ============================================================
# LOAD & VALIDATE GOVERNED INPUTS
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np


# Load governed datasets
PROJECT_ROOT = Path.cwd().parent.parent


DIM_PRODUCT_PATH = PROJECT_ROOT / "data" / "processed" / "dim_product.csv"
SALES_PATH = PROJECT_ROOT / "data" / "processed" / "fact_sales.csv"
SOA_PATH = PROJECT_ROOT / "data" / "processed" / "fact_soa.csv"
STOCK_PATH = PROJECT_ROOT / "data" / "processed" / "fact_stock.csv"


dim_product = pd.read_csv(DIM_PRODUCT_PATH )
fact_sales = pd.read_csv(SALES_PATH)
fact_soa = pd.read_csv(SOA_PATH)
fact_stock = pd.read_csv(STOCK_PATH)

datasets = {
    "dim_product": dim_product,
    "fact_sales": fact_sales,
    "fact_stock": fact_stock,
    "fact_soa": fact_soa
}

print("CROSS-FUNCTIONAL GOVERNED INPUTS")
print("=" * 80)

for name, df in datasets.items():
    print(
        f"{name:<15} "
        f"Rows: {len(df):>6,} | "
        f"Product_ID: {'Product_ID' in df.columns}"
    )

print()
print("PRODUCT POPULATIONS")
print("=" * 80)

for name, df in datasets.items():
    products = (
        df["Product_ID"].nunique()
        if "Product_ID" in df.columns
        else 0
    )

    print(f"{name:<15}: {products:,}")

print()
print("VALIDATION")
print("=" * 80)

all_have_product_id = all(
    "Product_ID" in df.columns
    for df in datasets.values()
)

print("Product_ID available in all datasets:", all_have_product_id)

CROSS-FUNCTIONAL GOVERNED INPUTS
dim_product     Rows:  1,436 | Product_ID: True
fact_sales      Rows:    966 | Product_ID: True
fact_stock      Rows:  2,681 | Product_ID: True
fact_soa        Rows:    412 | Product_ID: True

PRODUCT POPULATIONS
dim_product    : 1,436
fact_sales     : 767
fact_stock     : 383
fact_soa       : 412

VALIDATION
Product_ID available in all datasets: True


### Step 1 — Build the Product-Level Cross-Functional Dataset.

In [2]:
# ------------------------------------------------------------
# 1. Product base
# ------------------------------------------------------------

product_base = (
    dim_product[
        ["Product_ID", "Product_Key", "Product_Description",
         "Product_Category"]
    ]
    .drop_duplicates(subset=["Product_ID"])
    .copy()
)

# ------------------------------------------------------------
# 2. Aggregate Sales → Product_ID
# ------------------------------------------------------------

sales_product = (
    fact_sales
    .groupby("Product_ID", as_index=False)
    .agg(
        Sales_Months=("Source_Month", "nunique"),
        Net_Units=("Sold Period", "sum"),
        Revenue=("Sales Value", "sum"),
        Cost_Sales=("Cost Sales", "sum"),
        Gross_Profit=("Profit", "sum")
    )
)

# ------------------------------------------------------------
# 3. Aggregate Stock → Product_ID
# ------------------------------------------------------------

stock_product = (
    fact_stock
    .groupby("Product_ID", as_index=False)
    .agg(
        Stock_Units=("Quantity", "sum"),
        Outstanding_Order_Qty=("Outstanding_Order_Qty", "sum"),
        Stores_With_Stock=(
            "Quantity",
            lambda x: (x > 0).sum()
        )
    )
)

# ------------------------------------------------------------
# 4. Aggregate SOA → Product_ID
# ------------------------------------------------------------

soa_product = (
    fact_soa
    .groupby("Product_ID", as_index=False)
    .agg(
        SOA_Value=("SOA", "sum"),
        SOA_Start=("Starts", "min"),
        SOA_End=("Ends", "max")
    )
)

# ------------------------------------------------------------
# 5. Combine all domains
# ------------------------------------------------------------

cross_functional = (
    product_base
    .merge(sales_product, on="Product_ID", how="left")
    .merge(stock_product, on="Product_ID", how="left")
    .merge(soa_product, on="Product_ID", how="left")
)

# ------------------------------------------------------------
# 6. Presence flags
# ------------------------------------------------------------

cross_functional["Has_Sales"] = (
    cross_functional["Product_ID"]
    .isin(fact_sales["Product_ID"])
)

cross_functional["Has_Stock"] = (
    cross_functional["Product_ID"]
    .isin(fact_stock["Product_ID"])
)

cross_functional["Has_SOA"] = (
    cross_functional["Product_ID"]
    .isin(fact_soa["Product_ID"])
)

# ------------------------------------------------------------
# 7. Fill structural numeric gaps
# ------------------------------------------------------------

numeric_cols = [
    "Sales_Months",
    "Net_Units",
    "Revenue",
    "Cost_Sales",
    "Gross_Profit",
    "Stock_Units",
    "Outstanding_Order_Qty",
    "Stores_With_Stock",
    "SOA_Value"
]

cross_functional[numeric_cols] = (
    cross_functional[numeric_cols].fillna(0)
)

# ------------------------------------------------------------
# 8. Validation
# ------------------------------------------------------------

print("CROSS-FUNCTIONAL PRODUCT DATASET")
print("=" * 80)

print(f"Products              : {len(cross_functional):,}")
print(f"Unique Product_IDs     : {cross_functional['Product_ID'].nunique():,}")
print(f"Products with sales    : {cross_functional['Has_Sales'].sum():,}")
print(f"Products with stock    : {cross_functional['Has_Stock'].sum():,}")
print(f"Products with SOA      : {cross_functional['Has_SOA'].sum():,}")

print()
print("RECONCILIATION")
print("=" * 80)

print(
    "Product population:",
    len(cross_functional) == dim_product["Product_ID"].nunique()
)

print(
    "Sales revenue:",
    np.isclose(
        cross_functional["Revenue"].sum(),
        fact_sales["Sales Value"].sum()
    )
)

print(
    "Stock units:",
    np.isclose(
        cross_functional["Stock_Units"].sum(),
        fact_stock["Quantity"].sum()
    )
)

print(
    "SOA value:",
    np.isclose(
        cross_functional["SOA_Value"].sum(),
        fact_soa["SOA"].sum()
    )
)

display(cross_functional.head(10))

CROSS-FUNCTIONAL PRODUCT DATASET
Products              : 1,436
Unique Product_IDs     : 1,436
Products with sales    : 767
Products with stock    : 383
Products with SOA      : 412

RECONCILIATION
Product population: True
Sales revenue: True
Stock units: True
SOA value: True


,Product_ID,Product_Key,Product_Description,Product_Category,Sales_Months,Net_Units,Revenue,Cost_Sales,Gross_Profit,Stock_Units,Outstanding_Order_Qty,Stores_With_Stock,SOA_Value,SOA_Start,SOA_End,Has_Sales,Has_Stock,Has_SOA
0,1,010-02384-10,Garmin Lily Cream Gold & White,FITNESS,1.0,1.0,165.83,123.97,41.86,0.0,0.0,0.0,0.0,NaN,NaN,True,False,False
1,2,010-02784-00,Garmin Venu 3 Smartwatch - Silver,FITNESS,1.0,1.0,307.50,269.48,38.02,0.0,0.0,0.0,0.0,NaN,NaN,True,False,False
2,3,010-02784-01,Garmin Venu 3 Smartwatch - Slate,FITNESS,1.0,1.0,290.83,269.51,21.32,0.0,0.0,0.0,0.0,NaN,NaN,True,False,False
3,4,010-02839-00,"Garmin Lily 2, Cream Gold w/",FITNESS,1.0,1.0,165.83,150.92,14.91,0.0,0.0,0.0,0.0,NaN,NaN,True,False,False
4,5,01950,NUTRIBULLET PRO 4pc Starter Kit,BLENDERS,1.0,1.0,57.50,45.90,11.60,0.0,0.0,0.0,0.0,NaN,NaN,True,False,False
5,6,10009310,Miele GGRP Gourmet Griddle Plate,WHITES ACCESSORIES,1.0,1.0,83.33,130.96,-47.63,0.0,0.0,0.0,0.0,NaN,NaN,True,False,False
6,7,10107860,Miele SF-AP 50 Air Clean Plus Filter,VACUUM ACCESSORIES,1.0,1.0,15.00,11.41,3.59,0.0,0.0,0.0,0.0,NaN,NaN,True,False,False
7,8,10234470,Miele Nature Flacon,WHITES ACCESSORIES,1.0,1.0,9.16,5.01,4.15,0.0,0.0,0.0,0.0,NaN,NaN,True,False,False
8,9,102785,"MR Equip Metallic Red, 1.5 litre, 3kw",KETTLES,1.0,1.0,21.67,0.00,21.67,0.0,0.0,0.0,0.0,NaN,NaN,True,False,False
9,10,103414675,ZAGG Pro Keys 2-Apple iPad Pro 13,IT ACCESSORIES,1.0,1.0,82.50,76.17,6.33,0.0,0.0,0.0,0.0,NaN,NaN,True,False,False


### Step 2 — Cross-Functional Coverage & Overlap Analysis

In [4]:
# ------------------------------------------------------------
# 1. Coverage classification
# ------------------------------------------------------------

def classify_coverage(row):

    if row["Has_Sales"] and row["Has_Stock"] and row["Has_SOA"]:
        return "SALES_STOCK_SOA"

    elif row["Has_Sales"] and row["Has_Stock"]:
        return "SALES_STOCK"

    elif row["Has_Sales"] and row["Has_SOA"]:
        return "SALES_SOA"

    elif row["Has_Stock"] and row["Has_SOA"]:
        return "STOCK_SOA"

    elif row["Has_Sales"]:
        return "SALES_ONLY"

    elif row["Has_Stock"]:
        return "STOCK_ONLY"

    elif row["Has_SOA"]:
        return "SOA_ONLY"

    else:
        return "NO_ACTIVITY"


cross_functional["Coverage_Segment"] = (
    cross_functional.apply(classify_coverage, axis=1)
)

# ------------------------------------------------------------
# 2. Coverage summary
# ------------------------------------------------------------

coverage_summary = (
    cross_functional
    .groupby("Coverage_Segment", as_index=False)
    .agg(
        Products=("Product_ID", "nunique")
    )
)

coverage_summary["Product_Share_%"] = (
    coverage_summary["Products"]
    / cross_functional["Product_ID"].nunique()
    * 100
)

coverage_summary = (
    coverage_summary
    .sort_values("Products", ascending=False)
    .reset_index(drop=True)
)

display_summary = coverage_summary.copy()

display_summary["Product_Share_%"] = (
    display_summary["Product_Share_%"]
    .map(lambda x: f"{x:.2f}%")
)

print("CROSS-FUNCTIONAL PRODUCT COVERAGE")
print("=" * 80)

display(display_summary)

# ------------------------------------------------------------
# 3. Key functional overlaps
# ------------------------------------------------------------

sales_stock = (
    cross_functional["Has_Sales"]
    & cross_functional["Has_Stock"]
).sum()

sales_soa = (
    cross_functional["Has_Sales"]
    & cross_functional["Has_SOA"]
).sum()

stock_soa = (
    cross_functional["Has_Stock"]
    & cross_functional["Has_SOA"]
).sum()

all_three = (
    cross_functional["Has_Sales"]
    & cross_functional["Has_Stock"]
    & cross_functional["Has_SOA"]
).sum()

print("\nKEY CROSS-FUNCTIONAL OVERLAPS")
print("=" * 80)

print(f"Sales × Stock products : {sales_stock:,}")
print(f"Sales × SOA products   : {sales_soa:,}")
print(f"Stock × SOA products   : {stock_soa:,}")
print(f"Sales × Stock × SOA    : {all_three:,}")

# ------------------------------------------------------------
# 4. Validation
# ------------------------------------------------------------

classified_products = coverage_summary["Products"].sum()

print("\nVALIDATION")
print("=" * 80)

print(f"Products classified : {classified_products:,}")
print(
    f"Expected products   : "
    f"{cross_functional['Product_ID'].nunique():,}"
)

print(
    "Population reconciles:",
    classified_products
    == cross_functional["Product_ID"].nunique()
)

print(
    "Coverage flags valid:",
    cross_functional["Has_Sales"].sum() == 767
    and cross_functional["Has_Stock"].sum() == 383
    and cross_functional["Has_SOA"].sum() == 412
)

CROSS-FUNCTIONAL PRODUCT COVERAGE


,Coverage_Segment,Products,Product_Share_%
0,SALES_ONLY,649,45.19%
1,SOA_ONLY,342,23.82%
2,STOCK_ONLY,319,22.21%
3,SALES_SOA,62,4.32%
4,SALES_STOCK,56,3.90%
5,STOCK_SOA,8,0.56%



KEY CROSS-FUNCTIONAL OVERLAPS
Sales × Stock products : 56
Sales × SOA products   : 62
Stock × SOA products   : 8
Sales × Stock × SOA    : 0

VALIDATION
Products classified : 1,436
Expected products   : 1,436
Population reconciles: True
Coverage flags valid: True


### Step 3 — Cross-Functional Decision View

In [5]:
# ============================================================
# STEP 4.1 — SALES × STOCK DECISION SUMMARY
# ============================================================

sales_stock_view = cross_functional[
    cross_functional["Has_Sales"]
    & cross_functional["Has_Stock"]
].copy()

print("SALES × STOCK DECISION VIEW")
print("=" * 80)

print(f"Products analysed : {len(sales_stock_view):,}")
print(f"Net sales units   : {sales_stock_view['Net_Units'].sum():,.0f}")
print(f"Stock units       : {sales_stock_view['Stock_Units'].sum():,.0f}")

print("\nINVENTORY POSITION")
print("=" * 80)

print(
    f"Positive demand products : "
    f"{(sales_stock_view['Net_Units'] > 0).sum():,}"
)

print(
    f"No positive demand       : "
    f"{(sales_stock_view['Net_Units'] <= 0).sum():,}"
)

print(
    f"Stock with zero demand   : "
    f"{((sales_stock_view['Stock_Units'] > 0) & (sales_stock_view['Net_Units'] <= 0)).sum():,}"
)

SALES × STOCK DECISION VIEW
Products analysed : 56
Net sales units   : 75
Stock units       : 871

INVENTORY POSITION
Positive demand products : 55
No positive demand       : 1
Stock with zero demand   : 1


In [6]:
# ============================================================
# STEP 4.2 — SALES × SOA DECISION SUMMARY
# ============================================================

sales_soa_view = cross_functional[
    cross_functional["Has_Sales"]
    & cross_functional["Has_SOA"]
].copy()

print("SALES × SOA DECISION VIEW")
print("=" * 80)

print(f"Products analysed : {len(sales_soa_view):,}")
print(f"Net sales units   : {sales_soa_view['Net_Units'].sum():,.0f}")
print(f"Sales revenue     : £{sales_soa_view['Revenue'].sum():,.2f}")
print(f"Gross profit      : £{sales_soa_view['Gross_Profit'].sum():,.2f}")
print(f"SOA value         : £{sales_soa_view['SOA_Value'].sum():,.2f}")

print("\nCOMMERCIAL POSITION")
print("=" * 80)

print(
    f"Profitable products     : "
    f"{(sales_soa_view['Gross_Profit'] > 0).sum():,}"
)

print(
    f"Loss-making products    : "
    f"{(sales_soa_view['Gross_Profit'] < 0).sum():,}"
)

print(
    f"Zero-profit products    : "
    f"{(sales_soa_view['Gross_Profit'] == 0).sum():,}"
)

SALES × SOA DECISION VIEW
Products analysed : 62
Net sales units   : 104
Sales revenue     : £13,511.44
Gross profit      : £-1,669.71
SOA value         : £2,042.69

COMMERCIAL POSITION
Profitable products     : 27
Loss-making products    : 35
Zero-profit products    : 0


In [7]:
# ============================================================
# STEP 4.3 — VALIDATION
# ============================================================

print("CROSS-FUNCTIONAL VALIDATION")
print("=" * 80)

print(
    "Sales × Stock population:",
    len(sales_stock_view) == 56
)

print(
    "Sales × SOA population:",
    len(sales_soa_view) == 62
)

print(
    "Sales × Stock × SOA population:",
    (
        cross_functional["Has_Sales"]
        & cross_functional["Has_Stock"]
        & cross_functional["Has_SOA"]
    ).sum() == 0
)

CROSS-FUNCTIONAL VALIDATION
Sales × Stock population: True
Sales × SOA population: True
Sales × Stock × SOA population: True


### Step 5 — Cross-Functional Business Priority Summary

In [8]:
# ============================================================
# STEP 5.1 — CROSS-FUNCTIONAL BUSINESS ACTION
# ============================================================

def assign_business_action(row):

    # Sales + Stock:
    # inventory exists but no positive observed demand
    if (
        row["Has_Sales"]
        and row["Has_Stock"]
        and row["Stock_Units"] > 0
        and row["Net_Units"] <= 0
    ):
        return "INVENTORY_REVIEW"

    # Sales + SOA:
    # product is loss-making and has SOA support
    elif (
        row["Has_Sales"]
        and row["Has_SOA"]
        and row["Gross_Profit"] < 0
    ):
        return "PROFITABILITY_SOA_REVIEW"

    # Valid Sales + Stock relationship
    elif row["Has_Sales"] and row["Has_Stock"]:
        return "MONITOR_INVENTORY"

    # Valid Sales + SOA relationship
    elif row["Has_Sales"] and row["Has_SOA"]:
        return "MONITOR_COMMERCIAL"

    # No cross-functional overlap
    else:
        return "NO_CROSS_FUNCTIONAL_ACTION"


cross_functional["Business_Action"] = (
    cross_functional.apply(assign_business_action, axis=1)
)

In [9]:
# ============================================================
# STEP 5.2 — BUSINESS PRIORITY SUMMARY
# ============================================================

priority_summary = (
    cross_functional
    .groupby("Business_Action", as_index=False)
    .agg(
        Products=("Product_ID", "nunique")
    )
)

priority_summary["Product_Share_%"] = (
    priority_summary["Products"]
    / cross_functional["Product_ID"].nunique()
    * 100
).round(2)

priority_summary = (
    priority_summary
    .sort_values("Products", ascending=False)
    .reset_index(drop=True)
)

display_summary = priority_summary.copy()

display_summary["Product_Share_%"] = (
    display_summary["Product_Share_%"]
    .map(lambda x: f"{x:.2f}%")
)

print("CROSS-FUNCTIONAL BUSINESS PRIORITIES")
print("=" * 80)

display(display_summary)

CROSS-FUNCTIONAL BUSINESS PRIORITIES


,Business_Action,Products,Product_Share_%
0,NO_CROSS_FUNCTIONAL_ACTION,1318,91.78%
1,MONITOR_INVENTORY,55,3.83%
2,PROFITABILITY_SOA_REVIEW,35,2.44%
3,MONITOR_COMMERCIAL,27,1.88%
4,INVENTORY_REVIEW,1,0.07%


In [10]:
# ============================================================
# STEP 5.3 — PRODUCTS REQUIRING REVIEW
# ============================================================

review_products = cross_functional[
    cross_functional["Business_Action"].isin([
        "INVENTORY_REVIEW",
        "PROFITABILITY_SOA_REVIEW"
    ])
].copy()

review_products = review_products[
    [
        "Product_ID",
        "Product_Key",
        "Product_Description",
        "Product_Category",
        "Business_Action",
        "Net_Units",
        "Revenue",
        "Gross_Profit",
        "Stock_Units",
        "SOA_Value"
    ]
].sort_values(
    ["Business_Action", "Gross_Profit"],
    ascending=[True, True]
)

print("CROSS-FUNCTIONAL REVIEW PRODUCTS")
print("=" * 80)

display(review_products)

CROSS-FUNCTIONAL REVIEW PRODUCTS


,Product_ID,Product_Key,Product_Description,Product_Category,Business_Action,Net_Units,Revenue,Gross_Profit,Stock_Units,SOA_Value
379,380,DGE5861HM,AEG 80cm Canopy Cooker Hood,COOKER HOODS,INVENTORY_REVIEW,0.0,0.00,0.00,7.0,0.00
135,136,447033-01,Dyson V15 Detect Absolute,STICK VACS,PROFITABILITY_SOA_REVIEW,2.0,790.00,-239.16,0.0,66.50
316,317,BU3521UK,Shark Clean & Empty Cordless,STICK VACS,PROFITABILITY_SOA_REVIEW,3.0,497.49,-224.94,0.0,101.25
631,632,IZ400UKT,Shark Tru Pet Cordless Stick,STICK VACS,PROFITABILITY_SOA_REVIEW,2.0,381.67,-205.69,0.0,135.92
417,418,DZ400UK,Ninja MAX 6-in-1 Dual Zone Air Fryer,FRYERS,PROFITABILITY_SOA_REVIEW,5.0,660.01,-142.24,0.0,30.92
416,417,DZ300UK,Ninja 6-in-1 Dual Zone Air Fryer 7.6L,FRYERS,PROFITABILITY_SOA_REVIEW,3.0,299.15,-125.83,0.0,31.67
1119,1120,SL400UK,Ninja Double Stacked Air Fryer,FRYERS,PROFITABILITY_SOA_REVIEW,5.0,804.99,-101.56,0.0,22.58
630,631,IZ400UK,Shark Anti Hair Wrap Stick Vacuum,STICK VACS,PROFITABILITY_SOA_REVIEW,1.0,149.99,-97.92,0.0,136.67
137,138,454856-01,Dyson HP10 Hot & Cool Purifier StSt,HEATING,PROFITABILITY_SOA_REVIEW,1.0,299.17,-94.73,0.0,66.50
185,186,598961-01,Dyson Airstrait Rose Gold & Pink,HAIRCARE,PROFITABILITY_SOA_REVIEW,1.0,249.17,-82.94,0.0,99.75


In [11]:
# ============================================================
# STEP 5.4 — FINAL VALIDATION
# ============================================================

total_products = cross_functional["Product_ID"].nunique()

classified_products = priority_summary["Products"].sum()

inventory_review = (
    cross_functional["Business_Action"] == "INVENTORY_REVIEW"
).sum()

soa_review = (
    cross_functional["Business_Action"] == "PROFITABILITY_SOA_REVIEW"
).sum()


print("FINAL CROSS-FUNCTIONAL VALIDATION")
print("=" * 80)

print(f"Products classified        : {classified_products:,}")
print(f"Expected products          : {total_products:,}")
print(f"Inventory review products  : {inventory_review:,}")
print(f"Profit/SOA review products : {soa_review:,}")

print(
    "\nPopulation reconciles:",
    classified_products == total_products
)

print(
    "Business action complete:",
    cross_functional["Business_Action"].notna().all()
)

print(
    "Product_ID remains unique:",
    cross_functional["Product_ID"].is_unique
)

FINAL CROSS-FUNCTIONAL VALIDATION
Products classified        : 1,436
Expected products          : 1,436
Inventory review products  : 1
Profit/SOA review products : 35

Population reconciles: True
Business action complete: True
Product_ID remains unique: True


## Key Findings

- The governed cross-functional dataset contains **1,436 unique products**, with Sales, Stock and SOA measures successfully reconciled to their respective source datasets.

- Cross-functional coverage is limited:
  - **767 products** have sales history.
  - **383 products** have current stock records.
  - **412 products** have SOA records.

- Most products exist within only one functional dataset:
  - **649 (45.19%)** are Sales-only.
  - **342 (23.82%)** are SOA-only.
  - **319 (22.21%)** are Stock-only.

- Pairwise overlap remains relatively small:
  - **56 products** overlap between Sales and Stock.
  - **62 products** overlap between Sales and SOA.
  - **8 products** overlap between Stock and SOA.
  - **No products currently overlap across Sales, Stock and SOA simultaneously.**

- Within the **Sales × Stock** population:
  - 56 products were analysed.
  - These products hold **871 stock units** against **75 observed net sales units**.
  - 55 products have positive observed demand.
  - **1 product has stock but no positive observed demand** and is flagged for inventory review.

- Within the **Sales × SOA** population:
  - 62 products were analysed.
  - They generated **£13,511.44 in sales revenue**.
  - Aggregate gross profit was **-£1,669.71**.
  - **35 products are loss-making** and require profitability/SOA review.
  - 27 products are profitable and remain under commercial monitoring.

- The final business-action layer identified:
  - **1 INVENTORY_REVIEW**
  - **35 PROFITABILITY_SOA_REVIEW**
  - **55 MONITOR_INVENTORY**
  - **27 MONITOR_COMMERCIAL**
  - **1,318 NO_CROSS_FUNCTIONAL_ACTION**

- The absence of a three-way Sales × Stock × SOA overlap means that a fully integrated product-level commercial decision model is **not currently supported by the available data**. Pairwise analysis is therefore used without forcing unsupported relationships.

- Final validation passed: the **1,436-product population reconciles completely**, all products received a business-action classification, and `Product_ID` remained unique.

### Conclusion

The cross-functional analysis successfully establishes a governed product-level view across Sales, Stock and SOA while respecting the limitations of source overlap. It identifies a focused set of products requiring **inventory or profitability review** and provides a validated foundation for downstream reporting and decision-support analysis.